In [4]:
import pandas as pd
import numpy as np
df=pd.read_csv("DataSet_Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

X=df.drop(columns=["customerID","Churn"])
y = (df["Churn"] == "Yes").astype(int)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

numeric_feature = X.select_dtypes(include="number").columns
categorical_feature = X.select_dtypes(exclude="number").columns

numeric_pipeline=Pipeline ([
    ("impute",SimpleImputer(strategy="median")),
])
categorical_pipeline=Pipeline([
    ("impute",SimpleImputer(strategy="constant",fill_value="missing")),
     ("",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,numeric_feature),
    ("cat",categorical_pipeline,categorical_feature)
])


In [6]:
print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["Churn"].value_counts())
print("\nTarget proportions:")
print(df["Churn"].value_counts(normalize=True))

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nNumeric features:", len(numeric_feature))
print("Categorical features:", len(categorical_feature))

Dataset shape: (7043, 21)

Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Target proportions:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

Training shape: (5634, 19)
Test shape: (1409, 19)

Numeric features: 3
Categorical features: 16


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1035
           1       0.52      0.76      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.75      0.76      1409

Confusion matrix:
 [[778 257]
 [ 91 283]]
ROC-AUC: 0.8399364488878556


In [8]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_prob >= 0.5).astype(int)

print(classification_report(
    (y_test == "Yes").astype(int), xgb_pred
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "Yes").astype(int), xgb_prob
))
print("PR-AUC:", average_precision_score(
    (y_test == "Yes").astype(int), xgb_prob
))

              precision    recall  f1-score   support

           0       1.00      0.78      0.88      1409
           1       0.00      0.00      0.00         0

    accuracy                           0.78      1409
   macro avg       0.50      0.39      0.44      1409
weighted avg       1.00      0.78      0.88      1409

ROC-AUC: nan
PR-AUC: 0.0


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [9]:
print("y_train values:")
print(y_train.value_counts() if hasattr(y_train, "value_counts")
      else np.unique(y_train, return_counts=True))

print("\ny_test values:")
print(y_test.value_counts() if hasattr(y_test, "value_counts")
      else np.unique(y_test, return_counts=True))

print("\nXGBoost classes:", xgb_model.named_steps["classifier"].classes_)
print("Predicted probabilities:", xgb_prob[:5])
print("Predicted classes:", np.unique(xgb_pred, return_counts=True))


y_train values:
Churn
0    4139
1    1495
Name: count, dtype: int64

y_test values:
Churn
0    1035
1     374
Name: count, dtype: int64

XGBoost classes: [0 1]
Predicted probabilities: [0.0110268  0.8676396  0.07490411 0.30558386 0.00929246]
Predicted classes: (array([0, 1]), array([1104,  305]))


In [10]:
y_true = (y_test == "Yes").astype(int)

print("Actual churners:", y_true.sum())
print("Actual non-churners:", (y_true == 0).sum())

Actual churners: 0
Actual non-churners: 1409


In [11]:
y_true = y_test.astype(int)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

print(classification_report(y_true, xgb_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_true, xgb_prob))
print("PR-AUC:", average_precision_score(y_true, xgb_prob))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.53      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409

ROC-AUC: 0.8437637241985069
PR-AUC: 0.6592505616801314


In [13]:
import joblib
import os

os.makedirs("artifacts", exist_ok=True)

joblib.dump(
    xgb_model,
    "artifacts/telco_churn_xgboost.joblib"
)

print("Model saved successfully.")

Model saved successfully.


In [14]:
loaded_model = joblib.load(
    "artifacts/telco_churn_xgboost.joblib"
)

print("Sample churn probabilities:",
      loaded_model.predict_proba(X_test.iloc[:5])[:, 1])

Sample churn probabilities: [0.0110268  0.8676396  0.07490411 0.30558386 0.00929246]


In [15]:
import os
import joblib

os.makedirs("artifacts", exist_ok=True)

joblib.dump(
    xgb_model,
    "artifacts/telco_churn_xgboost.joblib"
)

loaded_model = joblib.load(
    "artifacts/telco_churn_xgboost.joblib"
)

print(loaded_model.predict_proba(X_test.iloc[:3])[:, 1])

[0.0110268  0.8676396  0.07490411]


In [16]:
print(df["TotalCharges"].dtype)

str


In [17]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_feature = X.select_dtypes(include="number").columns.tolist()
categorical_feature = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_feature),
    ("cat", categorical_pipeline, categorical_feature)
])

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

os.makedirs("artifacts", exist_ok=True)
joblib.dump(xgb_model, "artifacts/telco_churn_xgboost.joblib")

print("Corrected model trained and saved.")

Corrected model trained and saved.


In [18]:
import sys
!{sys.executable} -m pip install fastapi uvicorn


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


In [19]:

from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

MODEL_PATH = Path("artifacts/telco_churn_xgboost.joblib")

model = joblib.load(MODEL_PATH)
app = FastAPI(title="Customer Churn Prediction API")


class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid")

    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float


@app.get("/")
def home():
    return {"status": "running", "service": "churn-prediction"}


@app.post("/predict")
def predict(customer: Customer):
    try:
        X = pd.DataFrame([customer.model_dump()])
        probability = float(model.predict_proba(X)[0, 1])

        risk = (
            "High" if probability >= 0.70
            else "Medium" if probability >= 0.40
            else "Low"
        )

        return {
            "churn_probability": round(probability, 4),
            "risk_category": risk,
            "model_version": "1.0.0"
        }
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail="Prediction failed. Check model inputs and logs."
        ) from exc


In [20]:
%%writefile api.py
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

MODEL_PATH = Path("artifacts/telco_churn_xgboost.joblib")
model = joblib.load(MODEL_PATH)

app = FastAPI(title="Customer Churn Prediction API")

class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float

@app.get("/")
def home():
    return {"status": "running", "service": "churn-prediction"}

@app.post("/predict")
def predict(customer: Customer):
    try:
        X = pd.DataFrame([customer.model_dump()])
        probability = float(model.predict_proba(X)[0, 1])
        risk = (
            "High" if probability >= 0.70 
            else "Medium" if probability >= 0.40 
            else "Low"
        )
        return {
            "churn_probability": round(probability, 4),
            "risk_category": risk,
            "model_version": "1.0.0"
        }
    except Exception as exc:
        raise HTTPException(
            status_code=500, 
            detail="Prediction failed. Check model inputs and logs."
        ) from exc

Writing api.py


In [21]:
import uvicorn

if __name__ == "__main__":
    uvicorn.run("api:app", host="127.0.0.1", port=8000, reload=True)

INFO:     Will watch for changes in these directories: ['/Users/aryanshinde/Music/Projects/Customer Churn']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [97207] using StatReload
INFO:     Started server process [97253]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [97253]
INFO:     Stopping reloader process [97207]


In [22]:
import joblib

joblib.dump(xgb_model, "telco_churn_xgboost.joblib")

['telco_churn_xgboost.joblib']

In [23]:
%%writefile api.py
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, ConfigDict

MODEL_PATH = Path("telco_churn_xgboost.joblib")
model = joblib.load(MODEL_PATH)

app = FastAPI(title="Customer Churn API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)

class PredictionInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float

@app.get("/")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(data: PredictionInput):
    try:
        X = pd.DataFrame([data.model_dump()])
        probability = float(model.predict_proba(X)[0, 1])
        return {
            "churn_probability": round(probability, 4),
            "risk_category": (
                "High" if probability >= 0.70
                else "Medium" if probability >= 0.40
                else "Low"
            ),
        }
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail="Prediction failed; check model feature types."
        ) from exc

Overwriting api.py
